In [14]:
#Ollama dependencies
!sudo apt update
!sudo apt-get install zstd
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app
'sh' is not recognized as an internal or external command,
operable program or batch file.


In [15]:
%pip install pypdf
%pip install -U langchain langchain-text-splitters langchain-ollama langchain-unstructured langchain-community unstructured
%pip install faiss-cpu
%pip install unstructured[pdf]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Usuario\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Usuario\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Usuario\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Usuario\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [16]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [17]:
!ollama pull qwen2.5:7b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 2bada8a74506: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling 2f15b3218f05: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 


In [18]:
#Large language models (like llama3.1) and embedding models (like nomic-embed-text) must be downloaded to your machine separately before your Python scripts can access them
!ollama pull nomic-embed-text

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 


In [19]:
import os
import glob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
# Modern Ollama Imports
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [20]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# --- 1. Load Documents ---
def load_documents_from_directory(dir_path: str):
    # Find all files in the directory (e.g., PDFs)
    file_paths = glob.glob(f"{dir_path}/*.pdf*")

    if not file_paths:
        print(f"No files found in {dir_path}")
        return []

    print(f"Loading following files: {file_paths}")

    documents = []
    for file_path in file_paths:
        documents.extend(PyPDFLoader(file_path).load())
    return documents

# --- 2. Build or Load Vector Database ---
def create_vector_store(documents, embedding_model_name="nomic-embed-text"):
    # Initialize the modern Ollama embedding class
    embeddings = OllamaEmbeddings(model=embedding_model_name)

    # Split text into manageable chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)

    # Generate embeddings and populate FAISS database
    print(f"Generating embeddings for {len(chunks)} text chunks...")

    vector_store = FAISS.from_documents(chunks, embeddings)
    return vector_store

def format_docs(docs):
    formatted_docs = "\n\n".join(doc.page_content for doc in docs)

    print(f"\n------Retrieve docs:")
    for doc in docs:
      print(doc.page_content)
      print("\n")
    print("\n To be used as context for the RAG query.\n")

    return formatted_docs

# --- 3. Execute RAG Query Loop ---
def run_rag_pipeline(vector_store, model_name="qwen2.5:7b"):
    # Instantiate the modern ChatOllama class for generation
    llm = ChatOllama(model=model_name, temperature=0)

    # Define a retriever from our FAISS index
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    while True:
        print("\nAsk a question about your documents (or type 'exit'): ")
        query = input("Question: ")
        if query.lower() == 'exit':
            break

        # Retrieve relevant text context
        # relevant_docs = retriever.invoke(query)
        # format_docs = "\n\n".join([doc.page_content for doc in relevant_docs])

        #Ajusta el prompt para que el modelo genere respuestas basadas en el contexto recuperado
        prompt_instance = ChatPromptTemplate.from_template("""
            You are an expert assistant in aircraft accidents and aviation safety.
            Respond ONLY using information from the provided context.
            If the answer is not in the context, say: "I do not have that information in the document."
            Do not invent data, dates, or figures, do not make things up. Always respond in Spanish.

            Context:
            {context}

            Question: {question}
            Answer:
            """)

        rag_chain = (
                {
                    "context": itemgetter("question") | retriever | format_docs,
                    "question": itemgetter("question")
                }
                | prompt_instance
                | llm
                | StrOutputParser()
        )

        # Generate and stream the response from Ollama
        print("\n--- Answer ---")
        for chunk in rag_chain.stream({"question": query}):
            print(chunk, end="", flush=True)
        print("\n--------------")

In [21]:
# --- Main Execution ---
if __name__ == "__main__":
    DOCS_DIR = "rag_pdf_files"

    # 1. Ingest
    raw_docs = load_documents_from_directory(DOCS_DIR)

    if raw_docs:
        # 2. Embed & Index (Make sure to run: 'ollama pull nomic-embed-text' first)
        db = create_vector_store(raw_docs, embedding_model_name="nomic-embed-text")
        print("Index ready. You can now ask questions.")

        # 3. Query (Make sure to run: 'ollama pull llama3.2' first)
        run_rag_pipeline(db, model_name="qwen2.5:7b")
    else:
        print("No documents found to process.")

Loading following files: ['rag_pdf_files\\Desastre aéreo de Tenerife.pdf']
Generating embeddings for 33 text chunks...
Index ready. You can now ask questions.

Ask a question about your documents (or type 'exit'): 

--- Answer ---

------Retrieve docs:
Las dos aeronaves habían aterrizado en Los Rodeos ese mismo domingo y se encontraban entre 
varias aeronaves desviadas a Los Rodeos debido a la explosión de una bomba en su destino 
previsto, el Aeropuerto de Gran Canaria. Los Rodeos se había congestionado con aviones 
estacionados que bloqueaban la única calle de rodaje, lo que obligaba a las aeronaves que 
despegaban a rodar por la pista. Una densa niebla se extendía por el aeródromo, lo que reducía 
considerablemente la visibilidad para los pilotos y la torre de control. 
Para Pan Am fue el peor siniestro aéreo que involucró una aeronave estadounidense, mucho más 
que el vuelo 191 de American Airlines ocurrido dos años después. Para la KLM fue el siniestro más 
mortífero de una aerona